# 07 — Empirical routing: measured F_ij as edge weight

**Question.** Our `select_best_tree` ranks chains by a calibration-derived cost
(1 - F_CZ + decoherence + 1Q error). That predicts edge quality from RB
numbers reported by the device. Does *measuring* the entanglement quality
of every native CZ pair directly — and then picking the chain that maximises
the measured product — actually do better?

**Protocol** (`src.diagnostics.edge_bell_map`).

On every native CZ edge $(i,j)$ we prepare the 2-qubit graph state
$|G_{ij}\rangle = \mathrm{CZ}_{ij}\,|+\rangle|+\rangle$ and measure the three
stabilizers of that Bell-equivalent state:

$$F_{ij} = \tfrac{1}{4}\!\left(1 + \langle X_i Z_j\rangle + \langle Z_i X_j\rangle + \langle Y_i Y_j\rangle\right).$$

$F_{ij}>1/2$ certifies entanglement on that pair. Edges are grouped into
matchings via greedy edge coloring so each matching is one parallel circuit
— so the full 54-edge Emerald map costs only $\sim 4$ matchings $\times$ 3 bases
$=\sim 12$ jobs, not 162.

**Use as cost.** We then plug the measured $\{F_{ij}\}$ into the same multi-start
Prim's MST as before, with edge weight $w(i,j)=1-F_{ij}$. Compare against
the calibration tree on the same target $n$.

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

if not os.environ.get("IQM_TOKEN"):
    for p in [Path.cwd() / ".secrets" / "iqm_api_key",
              Path.cwd().parent / ".secrets" / "iqm_api_key"]:
        if p.exists():
            os.environ["IQM_TOKEN"] = p.read_text().strip()
            break

from src.backend import get_backend, select_best_tree, select_best_tree_empirical
from src.diagnostics import run_edge_map, fidelity_map, plot_edge_map
from src.diagnostics.edge_bell_map import fidelity_heatmap
from src.circuits.graph_state import build_graph_state
from src.witnesses.gme_graph import build_gme_circuits_graph, compute_gme_witness_graph, fidelity_lower_bound
from src.mitigation.parity_qrem import correction_factors_from_metrics, apply_qrem_to_stabilizers_graph
from qiskit import transpile

DEVICE = "garnet"   # smaller -> cheaper full edge map
TARGET_N = 12
MAP_SHOTS = 4000
GME_SHOTS = 8000

backend = get_backend(device=DEVICE)
print("Backend:", backend, " qubits:", backend.num_qubits)

## 1. Measure F_ij on every native CZ edge

In [ ]:
results = run_edge_map(backend, shots=MAP_SHOTS)
F = fidelity_map(results)
Fs = np.array(list(F.values()))
print(f"\nedges measured: {len(F)}")
print(f"  F: min={Fs.min():.3f}  mean={Fs.mean():.3f}  max={Fs.max():.3f}")
n_3sigma = sum(1 for r in results if r.entangled_3sigma)
print(f"  certified entangling at 3sigma: {n_3sigma}/{len(results)}")
weakest = sorted(results, key=lambda r: r.F)[:5]
print("  weakest 5 edges:")
for r in weakest:
    print(f"    {r.edge}: F={r.F:.3f}  z={r.z_score:.1f}")

with open(f"empirical_edge_map_{DEVICE}.json", "w") as f:
    json.dump([r.to_dict() for r in results], f, indent=2)

In [ ]:
plot_edge_map(results, backend=backend, save_path=f"empirical_edge_map_{DEVICE}.png",
              title=f"Measured graph-state fidelity on every native CZ pair ({DEVICE})")
fidelity_heatmap(results, backend, save_path=f"empirical_fidelity_heatmap_{DEVICE}.png")
plt.show()

## 2. Two routing strategies on the same target_n

- **Calibration tree** — `select_best_tree`, weights from RB / T1 / T2 / 1Q.
- **Empirical tree** — `select_best_tree_empirical`, weights $1 - F_{ij}^{\text{measured}}$.

In [ ]:
tree_cal = select_best_tree(backend, TARGET_N)
tree_emp = select_best_tree_empirical(backend, TARGET_N, F)

def edge_F_list(tree):
    out = []
    for (a, b) in tree["edges"]:
        out.append(F.get((min(a, b), max(a, b)), float("nan")))
    return np.array(out)

for name, t in (("calibration", tree_cal), ("empirical", tree_emp)):
    fs = edge_F_list(t)
    print(f"\n{name} tree on {TARGET_N} qubits")
    print(f"  qubits: {t['qubits']}")
    print(f"  total weight: {t['weight']:.4f}")
    print(f"  edge F (measured)  min={np.nanmin(fs):.3f}  mean={np.nanmean(fs):.3f}")

## 3. Head-to-head: GME witness W on each tree

In [ ]:
def run_W(tree, shots=GME_SHOTS):
    n = len(tree["qubits"])
    state = build_graph_state(n, tree["logical_edges"])
    ca, cb = build_gme_circuits_graph(state, tree["coloring"])
    tr = transpile([ca, cb], backend=backend, initial_layout=tree["qubits"], optimization_level=3)
    res = backend.run(tr, shots=shots).result()
    counts_a = res.get_counts(0); counts_b = res.get_counts(1)
    raw = compute_gme_witness_graph(counts_a, counts_b, n, tree["logical_edges"], tree["coloring"])

    # Parity QREM
    cfactors = correction_factors_from_metrics(backend, tree["qubits"])
    g_mit = apply_qrem_to_stabilizers_graph(
        counts_a, counts_b, n, tree["logical_edges"], tree["coloring"], cfactors)
    W_mit = sum(g_mit.values())
    fb = fidelity_lower_bound(counts_a, counts_b, n, tree["logical_edges"], tree["coloring"])
    return {"raw_W": raw["W"], "raw_sigma": raw["sigma_W"],
            "sig_raw": raw["significance_sigma"],
            "W_qrem": W_mit, "F_lb": fb["F_lower_bound"]}

out_cal = run_W(tree_cal)
out_emp = run_W(tree_emp)

print(f"\n{'tree':<14}{'W_raw':>9}{'sigma':>10}{'W_qrem':>11}{'F_lb':>9}")
for label, o in (("calibration", out_cal), ("empirical", out_emp)):
    print(f"  {label:<12} {o['raw_W']:>7.3f} {o['sig_raw']:>+8.1f}sig {o['W_qrem']:>9.3f} {o['F_lb']:>7.3f}")

biseparable_bound = TARGET_N - 1
print(f"\nbiseparable bound: W <= {biseparable_bound}  (anything above => GME)")

In [ ]:
labels = ["calibration\ncost", "empirical\nF_ij"]
Wraw = [out_cal["raw_W"], out_emp["raw_W"]]
Wmit = [out_cal["W_qrem"], out_emp["W_qrem"]]

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(2); w = 0.36
ax.bar(x - w/2, Wraw, w, label="raw W", color="#4c72b0")
ax.bar(x + w/2, Wmit, w, label="QREM W", color="#55a868")
ax.axhline(biseparable_bound, color="red", linestyle="--", label=f"biseparable bound (n-1={biseparable_bound})")
ax.axhline(TARGET_N, color="gray", linestyle=":", label=f"ideal W = n = {TARGET_N}")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("W"); ax.set_title(f"GME witness on a {TARGET_N}-qubit chain ({DEVICE})\ncalibration cost vs measured F_ij")
ax.legend(loc="lower right")
fig.tight_layout(); fig.savefig(f"empirical_vs_calibration_W_{DEVICE}.png", dpi=120)
plt.show()

summary = {
    "device": DEVICE,
    "target_n": TARGET_N,
    "map_shots": MAP_SHOTS,
    "gme_shots": GME_SHOTS,
    "calibration": {**out_cal, "qubits": tree_cal["qubits"], "edges": tree_cal["edges"]},
    "empirical":   {**out_emp, "qubits": tree_emp["qubits"], "edges": tree_emp["edges"],
                     "min_edge_F_used": tree_emp["min_edge_F_used"],
                     "mean_edge_F_used": tree_emp["mean_edge_F_used"]},
}
with open(f"empirical_vs_calibration_{DEVICE}.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(json.dumps(summary, indent=2, default=float))

## Notes

- The empirical map costs **3 bases × #matchings** circuits, independent of
  the number of edges — for Garnet (~25 edges, ~4 matchings) that's ~12
  circuits before tree selection. Cheap.
- The two trees may pick *different qubit sets*, not just different edges:
  if calibration over-rates a coupler, empirical routing avoids it.
- The empirical cost only sees pair-level errors. It misses 1Q gate error,
  $T_1/T_2$, and crosstalk effects beyond the pair — so it can lose to
  calibration on devices whose dominant error is single-qubit. The honest
  takeaway is *which* tree wins, not which method is universally better.